In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,classification_report
import numpy as np

final_data_classification = pd.read_csv(r"C:\Users\20245179\OneDrive - TU Eindhoven\Ektha Thesis\FewShot\Few_shot_evaluation_data.csv")
final_data_classification["Comments"] = final_data_classification["Comments"].apply(lambda x: str(x))
final_data_classification['Comments'] = final_data_classification['Comments'].str.replace('"','')





c:\Users\20245179\AppData\Local\anaconda3\envs\thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
final_data_classification

,Unnamed: 0,Comments,Labels
0,0,Dit is echt een schande. In deze tijd van woni...,0
1,1,Waarom is Participe gewoon niet langer open. E...,0
2,2,Weer een woning uit het toch al beroerde besta...,0
3,3,in dat blok hebben wij gewoond. Geweldig huis!,0
4,4,Zonde om een woning hiervoor te gebruiken. Er ...,0
...,...,...,...
8577,8577,"Zeker, ik zal mijn contactgegevens actueel hou...",1
8578,8578,"Wow, dat klinkt als een hele onderneming! Als ...",1
8579,8579,Als ik terugdenk aan de heerlijke geur van ver...,1
8580,8580,"Wat een inspirerend voorbeeld, bedankt! Missch...",1


In [3]:
seed = 42
np.random.seed(seed)
 
train_texts, test_texts, train_labels, test_labels = train_test_split(final_data_classification["Comments"], final_data_classification["Labels"], test_size=.2,stratify=final_data_classification["Labels"])

train_texts, val_texts, train_labels, val_labels = train_test_split(train_texts, train_labels, test_size=.2,stratify=train_labels)
tokenizer = tokenizer = AutoTokenizer.from_pretrained("DTAI-KULeuven/robbert-2023-dutch-large")

def tokenize_function(examples):
    return tokenizer(examples,  padding="max_length", truncation=True, max_length=512, return_tensors="pt")

train_encodings = tokenize_function(list(train_texts))
val_encodings = tokenize_function(list(val_texts))
test_encodings = tokenize_function(list(test_texts))


# Prepare the dataset
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Use .clone().detach() to create a new tensor from existing tensors
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx]).clone().detach()
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = CustomDataset(train_encodings, list(train_labels.astype(int)))
test_dataset = CustomDataset(test_encodings, list(test_labels.astype(int)))
val_dataset = CustomDataset(val_encodings, list(val_labels.astype(int)))

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    #print("Labels",labels)
    #print("Preds",preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0.0)
    acc = accuracy_score(labels, preds)

    class_report = classification_report(labels, preds, target_names=['Real Data', 'Generated Data'], output_dict=True)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'classification_report': class_report
    }


model = AutoModelForSequenceClassification.from_pretrained("DTAI-KULeuven/robbert-2023-dutch-large",num_labels=2)

training_args = TrainingArguments(
        output_dir='./results',          # output directory
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=1,
        lr_scheduler_type="linear",
        learning_rate=1e-5,
        weight_decay=0.1,
        warmup_steps=200,
        max_grad_norm=1.0,
        logging_dir='./logs',
        evaluation_strategy="steps",
        logging_steps=200,
        gradient_accumulation_steps=1,
       
    )

trainer = Trainer(
        model=model,                         # the instantiated 🤗 Transformers model to be trained
        args=training_args,                  # training arguments, defined above
        train_dataset=train_dataset,       # training dataset
        eval_dataset = val_dataset,
      compute_metrics = compute_metrics
    )

# Train the model
trainer.train()
eval_output = trainer.evaluate(test_dataset)
print(eval_output)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at DTAI-KULeuven/robbert-2023-dutch-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\20245179\AppData\Local\anaconda3\envs\thesis\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
 15%|█▍        | 200/1373 [00:32<03:13,  6.06it/s]

{'loss': 0.3006, 'grad_norm': 0.08114340156316757, 'learning_rate': 1e-05, 'epoch': 0.15}


                                                  
 15%|█▍        | 201/1373 [00:47<1:26:21,  4.42s/it]

{'eval_loss': 0.05587195232510567, 'eval_accuracy': 0.9919883466860888, 'eval_f1': 0.9919877346504083, 'eval_precision': 0.9920112642341468, 'eval_recall': 0.9919736791848272, 'eval_classification_report': {'Real Data': {'precision': 0.9899135446685879, 'recall': 0.9942112879884226, 'f1-score': 0.992057761732852, 'support': 691.0}, 'Generated Data': {'precision': 0.9941089837997055, 'recall': 0.9897360703812317, 'f1-score': 0.9919177075679647, 'support': 682.0}, 'accuracy': 0.9919883466860888, 'macro avg': {'precision': 0.9920112642341468, 'recall': 0.9919736791848272, 'f1-score': 0.9919877346504083, 'support': 1373.0}, 'weighted avg': {'precision': 0.9919975137053121, 'recall': 0.9919883466860888, 'f1-score': 0.9919881936771688, 'support': 1373.0}}, 'eval_runtime': 14.1843, 'eval_samples_per_second': 96.797, 'eval_steps_per_second': 24.252, 'epoch': 0.15}


 18%|█▊        | 247/1373 [00:54<03:05,  6.06it/s]  

KeyboardInterrupt: 

In [ ]:
  FewSHot= {'eval_loss': 0.04798680916428566, 'eval_accuracy': 0.9924242424242424, 'eval_f1': 0.9924241781058476, 'eval_precision': 0.9924364268637252, 'eval_recall': 0.9924665303654999, 'eval_classification_report': {'Real Data': {'precision': 0.9976635514018691, 'recall': 0.9872832369942196, 'f1-score': 0.9924462521789658, 'support': 865.0}, 'Generated Data': {'precision': 0.9872093023255814, 'recall': 0.9976498237367802, 'f1-score': 0.9924021040327294, 'support': 851.0}, 'accuracy': 0.9924242424242424, 'macro avg': {'precision': 0.9924364268637252, 'recall': 0.9924665303654999, 'f1-score': 0.9924241781058476, 'support': 1716.0}, 'weighted avg': {'precision': 0.9924790724019154, 'recall': 0.9924242424242424, 'f1-score': 0.9924243581973533, 'support': 1716.0}}, 'eval_runtime': 17.6364, 'eval_samples_per_second': 97.299, 'eval_steps_per_second': 24.325, 'epoch': 1.0}
  ZeroShot